# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)
import pandas as pd
import numpy as np
import scipy.stats 
import statsmodels.api 

df = pd.read_csv("customers.csv")

df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

print("Missing values per column:")
print(df.isna().sum())

print("\nUnique region values:", df["region"].unique())

print("\nIncome summary:")
print(df["income"].describe())

# "east" and "East" should be the same category
df["region"] = df["region"].str.strip().str.title()
print("\nRegion values after fixing:", df["region"].unique())

# income can't be negative, so -50.0 is data entry error
df.loc[df["income"] < 0, "income"] = np.nan

# using the median instead of the mean because income is often skewed,
income_median = df["income"].median()
df["income"] = df["income"].fillna(income_median)

print("\nMissing values after cleaning:")
print(df.isna().sum())

print("\nIncome summary after cleaning:")
print(df["income"].describe())

print("\nFinal region categories:", df["region"].unique())

Missing values per column:
customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64

Unique region values: <StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str

Income summary:
count     38.000000
mean      83.007895
std       27.628563
min      -50.000000
25%       72.975000
50%       84.700000
75%       98.500000
max      114.500000
Name: income, dtype: float64

Region values after fixing: <StringArray>
['West', 'South', 'East']
Length: 3, dtype: str

Missing values after cleaning:
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64

Income summary after cleaning:
count     40.000000
mean      86.460000
std       16.080165
min       53.300000
25%       75.075000
50%       84.700000
75%       97.100000
max      114.500000
Name: income, dtype: float64

Final region categories: <StringArray>
['West', 'South', 'East']
Length: 3, dtype: str


**Explain each issue:** what you found and how you handled it.

*Your answer:*
1. The income column had two missing values (rows 4 and 18)
2. One region entry was 'east' instead of 'East' which pandas would treat as a seperate region 
3. Row 26 had an income of -50 which is impossible since income cant be negative 

To fix these, I standardized the region column's capitalization using .str.title(), converted the negative income to NaN, and filled all missing income values with the median income rather than the mean.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = continuous, spending = continuous
# tool: Pearson correlation (two continuous variables moving together)

corr, p_value = scipy.stats.pearsonr(df["income"], df["spending"])

print(f"Correlation between income and spending: r = {corr:.3f}")
print(f"p-value = {p_value:.4f}")

Correlation between income and spending: r = 0.717
p-value = 0.0000


*Your answer:* There is a strong, positive correlation between income and spending.


### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = categorical, subscribed = categorical
# tool: Chi-square test of independence (two categorical variables)

contingency_table = pd.crosstab(df["region"], df["subscribed"])
print("Contingency table:")
print(contingency_table)

chi2, p_value, dof, expected = scipy.stats.chi2_contingency(
    contingency_table,
    correction=False
)

print(f"\nChi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p_value:.4f}")

# check the assumption
print("\nExpected frequencies:")
print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns).round(2))

Contingency table:
subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Chi-square = 13.162
Degrees of freedom = 2
p-value = 0.0014

Expected frequencies:
subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0


*Your answer:* There is a statistically significant relationship between region and subscription status (χ^2 = 13.162, df = 2, p = 0.0014), meaning subscription rates do differ by region, customers in the East are far more likely to subscribe (16 of 21) than customers in the South (3 of 9) or West (1 of 10).


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: outcome = spending (continuous), predictors = income and age (both continuous)
# tool: Multiple linear regression (two predictors instead of one)

X = df[["income", "age"]]
X = statsmodels.api.add_constant(X)  # adds the intercept term
y = df["spending"]

model = statsmodels.api.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        14:26:56   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:* income and age together explain about 52% of the variation in spending (R^2 = 0.520). Income is a significant predictor of spending (coefficient =0.359, p =0.003), meaning each additional unit of income is associated with higher spending even after accounting for age. Age is not a significant predictor of spending, so once income is accounted for, age has little independent effect on how much a customer spends. 


### Q4. Which region has the most customers, and what is the average income overall?

In [6]:
# data types: Region = categorical, income = continuous
# Tool: simple descriptive statistics, no hypothesis test needed for this 

# Count how many customers are in each region
region_counts = df["region"].value_counts()
print("Customers per region:")
print(region_counts)

# Average income overall (using the cleaned data from Part 0)
average_income = df["income"].mean()
print(f"\nAverage income overall: {average_income:.2f}")

Customers per region:
region
East     21
West     10
South     9
Name: count, dtype: int64

Average income overall: 86.46


*Your answer:* The East region has the most customers (21 out of 40, more than half the total), and the average income across all customers is 86.46 (likely in thousands of dollars, so roughly $86,460).


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* In each case, the tool was determined by the data types involved. two continuous variables pointed to correlation (Q1), two categorical variables pointed to a chi-square test (Q2), one continuous outcome with multiple continuous predictors pointed to multiple regression (Q3), and a single variable with no comparison at all just needed basic descriptive statistics (Q4)


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* no we cant say income causes higher spending. Correlation only tells us that income and spending move together, not that one directly causes the other. The relationship could be because of a completely different thing (for example, age, occupation, or some other variable not in the data), or spending could even be influencing how income is reported, and theres no way to tell from a correlation alone


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [7]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)
print("Expected frequencies:")
print(expected_df)

print("\nAll expected counts >= 5?", (expected_df >= 5).all().all())

Expected frequencies:
subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0

All expected counts >= 5? False


*Your answer:* Not all expected counts meet the >= 5 rule, two cells (South) are at 4.5. The result is probably still real since the pattern is strong and the p-value is small, but with only 40 customers, the company should treat this with some caution and collect more data before relying on it fully.


### Push your code to GitHub